<a href="https://colab.research.google.com/github/IssacG1/Control-Compras-JAVA/blob/main/Copia_de_Hands_On_Fine_Tuning_con_LoRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **HANDS - ON: FINE-TUNING CON LORA**

Una vez vista la masterclass ***Fine-tuning y Evaluación de Modelos***, se proporciona el siguiente ***Colab*** para ejecutar, en vivo, un fine-tuning real con LoRA sobre un modelo Llama ligero, y medir su mejora con una métrica objetiva.

A diferencia de los Temas anteriores, aquí no usamos Groq — Groq solo sirve para inferencia, no para entrenar modelos. Usamos **Hugging Face** (librerías `transformers` y `peft`) directamente sobre la GPU gratuita de Colab.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1cKZ_hCf231RE84FDvGkEiKMH6ZDkVZzH?usp=sharing).

## **CONFIGURACIÓN DEL ENTORNO**

### **COLAB SECRETS**

Para no exponer tu ***token*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarlo de forma segura, añadiendo un nombre asociado al token para guardarlo dentro de una variable y usarlo dentro del notebook. Para este Tema necesitas un ***token de Hugging Face*** (el modelo que usamos es de acceso libre, no requiere solicitar permiso especial).

In [1]:
# Instalar librerias e iniciar sesión en Hugging Face con el token desde Colab Secrets


In [3]:
# Instalar librerias e iniciar sesión en Hugging Face con el token desde Colab Secrets

!pip install transformers peft accelerate trl --quiet

import torch
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_API'))
print("Sesión de Hugging Face iniciada correctamente.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 10.4 MB/s eta 0:00:00
Sesión de Hugging Face iniciada correctamente.


### **CARGAR EL MODELO BASE**

Usamos un modelo Llama ligero (pocos parámetros) para que el fine-tuning corra en minutos sobre la GPU T4 gratuita de Colab, sin necesitar cuantización adicional.

In [2]:
# Cargar el modelo base de Llama y su tokenizer


In [5]:
# Cargar el modelo base de Llama y su tokenizer

from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging
logging.set_verbosity_error()

modelo_base = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# variante oficial de Meta "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(modelo_base)
modelo = AutoModelForCausalLM.from_pretrained(modelo_base, dtype=torch.float16, device_map="auto")
print("Modelo base cargado:", modelo_base)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Modelo base cargado: TinyLlama/TinyLlama-1.1B-Chat-v1.0


### **ANTES DEL FINE-TUNING: LÍNEA BASE**

Antes de ajustar nada, probamos el modelo base con un prompt de ejemplo para tener un punto de comparación. El modelo aún no conoce el tono ni el formato que le vamos a enseñar.

In [3]:
# Definir una función para generar texto y probar el modelo base con un prompt de ejemplo



In [6]:
# Definir una función para generar texto y probar el modelo base con un prompt de ejemplo

def generar_respuesta(modelo_a_usar, prompt, max_new_tokens=60):
    entrada = tokenizer(prompt, return_tensors="pt").to(modelo_a_usar.device)
    salida = modelo_a_usar.generate(
        **entrada,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        no_repeat_ngram_size=3,
    )
    tokens_nuevos = salida[0][entrada["input_ids"].shape[1]:]
    texto_generado = tokenizer.decode(tokens_nuevos, skip_special_tokens=True)
    return texto_generado.split("\n")[0].strip()

prompt_prueba = "Cliente: ¿Puedo cambiar mi pedido después de comermelo?\nAgente:"

respuesta_base = generar_respuesta(modelo, prompt_prueba)
print(respuesta_base)

No, no puedes cambiar tu pedido.


### **PREPARAR LOS DATOS DE ENTRENAMIENTO**

El fine-tuning necesita ejemplos de entrada y salida que muestren el comportamiento que queremos enseñarle al modelo. Con pocos ejemplos (5 a 10) es suficiente para una demo — no es un dataset de producción.

In [4]:
# Definir una lista de ejemplos (entrada -> respuesta esperada) y convertirla en dataset


In [7]:
# Definir una lista de ejemplos (entrada -> respuesta esperada) y convertirla en dataset

from datasets import Dataset

ejemplos = [
    {"texto": "Cliente: ¿Puedo cambiar mi pedido después de pagarlo?\nAgente: Sí, puedes "
     "solicitar el cambio dentro de la primera hora escribiendo a soporte@tienda.com."},
    {"texto": "Cliente: ¿Cuánto tarda el reembolso?\nAgente: El reembolso se refleja en un plazo de 5 a 7 días hábiles."},
    {"texto": "Cliente: ¿Tienen envío el mismo día?\nAgente: Sí, disponible en zonas seleccionadas si el pedido se confirma antes de las 12:00."},
    {"texto": "Cliente: ¿Puedo pagar en el momento de la entrega?\nAgente: Sí, aceptamos pago contra entrega en efectivo o tarjeta."},
    {"texto": "Cliente: ¿Cómo rastreo mi paquete?\nAgente: Puedes rastrearlo con el número de guía en la sección 'Mis pedidos' de tu cuenta."},
]

dataset = Dataset.from_list(ejemplos)
dataset

Dataset({
    features: ['texto'],
    num_rows: 5
})

In [8]:
# Definir una lista de ejemplos (entrada -> respuesta esperada) y convertirla en dataset

from datasets import Dataset

ejemplos = [
    {"texto": "Cliente: ¿Puedo cambiar mi pedido después de pagarlo?\nAgente: Sí, puedes "
     "solicitar el cambio dentro de la primera hora escribiendo a soporte@tienda.com."},
    {"texto": "Cliente: ¿Cuánto tarda el reembolso?\nAgente: El reembolso se refleja en un plazo de 5 a 7 días hábiles."},
    {"texto": "Cliente: ¿Tienen envío el mismo día?\nAgente: Sí, disponible en zonas seleccionadas si el pedido se confirma antes de las 12:00."},
    {"texto": "Cliente: ¿Puedo pagar en el momento de la entrega?\nAgente: Sí, aceptamos pago contra entrega en efectivo o tarjeta."},
    {"texto": "Cliente: ¿Cómo rastreo mi paquete?\nAgente: Puedes rastrearlo con el número de guía en la sección 'Mis pedidos' de tu cuenta."},
]

dataset = Dataset.from_list(ejemplos)
dataset

Dataset({
    features: ['texto'],
    num_rows: 5
})

## **REALIZAR FINE-TUNING**

### **CONFIGURAR Y APLICAR LORA**

LoRA agrega matrices pequeñas entrenables sin tocar los pesos originales del modelo — por eso es tan ligero comparado con un fine-tuning completo.

In [9]:
# Configurar LoRA (rango, alpha, módulos objetivo) y aplicarlo al modelo base
!pip uninstall -y torchao --quiet

from peft import LoraConfig, get_peft_model
from transformers import set_seed
set_seed(42)

config_lora = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.0,
    task_type="CAUSAL_LM"
)

modelo_lora = get_peft_model(modelo, config_lora)
modelo_lora.print_trainable_parameters()
# r más alto = más capacidad para aprender, pero también más parámetros entrenables

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


### **ENTRENAR CON LORA**

Con el dataset y LoRA ya configurados, ejecutamos el entrenamiento. La pérdida (*loss*) que reporta el entrenador es nuestra métrica objetiva: debería bajar a medida que el modelo aprende los ejemplos.

In [10]:
# Configurar el entrenador (SFTTrainer) y ejecutar el fine-tuning
from trl import SFTTrainer, SFTConfig

config_entrenamiento = SFTConfig(
    output_dir="/content/resultados",
    num_train_epochs=30,
    per_device_train_batch_size=5,
    learning_rate=2e-4,
    logging_steps=1,
    dataset_text_field="texto",
    max_length=128,
    report_to="none",
    fp16=True,
    bf16=False
)

trainer = SFTTrainer(
    model=modelo_lora,
    train_dataset=dataset,
    args=config_entrenamiento,
)

resultado_entrenamiento = trainer.train()
print("Pérdida final:", resultado_entrenamiento.training_loss)

Adding EOS to train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

{'loss': '2.572', 'grad_norm': '1.725', 'learning_rate': '0.0002', 'entropy': '2.256', 'mean_token_accuracy': '0.4891', 'num_tokens': '234', 'epoch': '1'}
{'loss': '2.521', 'grad_norm': '1.736', 'learning_rate': '0.0001933', 'entropy': '2.248', 'mean_token_accuracy': '0.4978', 'num_tokens': '468', 'epoch': '2'}
{'loss': '2.459', 'grad_norm': '1.894', 'learning_rate': '0.0001867', 'entropy': '2.24', 'mean_token_accuracy': '0.4978', 'num_tokens': '702', 'epoch': '3'}
{'loss': '2.388', 'grad_norm': '1.994', 'learning_rate': '0.00018', 'entropy': '2.23', 'mean_token_accuracy': '0.5153', 'num_tokens': '936', 'epoch': '4'}
{'loss': '2.317', 'grad_norm': '1.9', 'learning_rate': '0.0001733', 'entropy': '2.216', 'mean_token_accuracy': '0.524', 'num_tokens': '1170', 'epoch': '5'}
{'loss': '2.248', 'grad_norm': '1.943', 'learning_rate': '0.0001667', 'entropy': '2.197', 'mean_token_accuracy': '0.5328', 'num_tokens': '1404', 'epoch': '6'}
{'loss': '2.176', 'grad_norm': '2.118', 'learning_rate': '0.

### **DESPUÉS DEL FINE-TUNING: MEDIR LA MEJORA**

Compararemos la pérdida antes y después del entrenamiento como métrica objetiva.

In [11]:
# Comparar la pérdidas
perdida_inicial = trainer.state.log_history[0]['loss']
perdida_final = resultado_entrenamiento.training_loss

print(f"Pérdida al inicio del entrenamiento: {perdida_inicial:.2f}")
print(f"Pérdida final del entrenamiento: {perdida_final:.2f}")
print(f"Reducción: {(1 - perdida_final/perdida_inicial) * 100:.0f}%")

# trainer.state.log_history[0]['loss'] es la pérdida después del primer paso registrado,
# no la pérdida real del modelo sin ningún entrenamiento.

Pérdida al inicio del entrenamiento: 2.57
Pérdida final del entrenamiento: 1.77
Reducción: 31%


In [ ]:
# Referencia cualitativa vs Referencia con un modelo de producción



In [12]:
# Referencia cualitativa vs Referencia con un modelo de producción

!pip install groq -q
from groq import Groq

respuesta_ajustada = generar_respuesta(modelo_lora, prompt_prueba)
print("Respuesta del modelo ajustado (TinyLlama + LoRA):\n", respuesta_ajustada)

client = Groq(api_key=userdata.get('GROQ_API_KEY'))

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_prueba + " Respuesta muy breve y corta."}]
)

print("\nRespuesta de referencia (Groq / GPT-OSS-20B):\n", response.choices[0].message.content)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 3.5 MB/s eta 0:00:00
Respuesta del modelo ajustado (TinyLlama + LoRA):
 Surgery, and the world.

Respuesta de referencia (Groq / GPT-OSS-20B):
 No.


**Nota:** Por eso, además de la respuesta de TinyLlama, se muestra una respuesta de Groq (GPT-OSS-20B) para el mismo prompt: no porque el proceso de fine-tuning haya fallado, sino como punto de comparación de cómo respondería un modelo de producción con muchos más parámetros y datos de entrenamiento — TinyLlama con 5 ejemplos demuestra la técnica de LoRA, no busca igualar la calidad de un modelo así de grande.

## **CHALLENGE: AJUSTE DE TONO CON LORA**

Una vez visto el ***Hands-On: Fine-tuning con LoRA***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se ajustará un modelo Llama ligero con un dataset propio para enseñarle un tono o formato de respuesta específico, comparando la pérdida **antes** y **después** del fine-tuning como métrica objetiva. En esta solución se usa como ejemplo un asistente de dudas frecuentes del propio curso.

**IMPORTANTE:** Para su revisión, es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.

### **INSTRUCCIONES:**

**1. Carga el modelo y define tu dataset:**

   * Instala las librerías, inicia sesión en Hugging Face con tu token y carga el modelo base junto con la función `generar_respuesta`.

   * Construye una lista llamada `ejemplos` con al menos 4 pares de entrada/respuesta que reflejen el tono o formato que quieres enseñarle al modelo, y conviértela en `dataset`.

In [14]:
# Instalar librerias e iniciar sesión en Hugging Face con el token desde Colab Secrets
!pip install -q transformers peft trl datasets accelerate

from google.colab import userdata
from huggingface_hub import login

# Llama a tu token guardado previamente
token = userdata.get('HF_API')
login(token)

In [18]:
# Cargar el modelo base de Llama y su tokenizer
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging
logging.set_verbosity_error()

modelo_base = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# variante oficial de Meta "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(modelo_base)
modelo = AutoModelForCausalLM.from_pretrained(modelo_base, dtype=torch.float16, device_map="auto")
print("Modelo base cargado:", modelo_base)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Modelo base cargado: TinyLlama/TinyLlama-1.1B-Chat-v1.0


In [19]:
# Definir la funcion de generacion de texto
def generar_respuesta(prompt, modelo, tokenizer):
    inputs = tokenizer(prompt, return_tensors="pt").to(modelo.device)
    outputs = modelo.generate(**inputs, max_new_tokens=50)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [20]:
# Definir la lista ejemplos y convertirla en dataset
from datasets import Dataset

ejemplos = [
    {"texto": "Pregunta: ¿Qué es LoRA?\nRespuesta: Es una técnica eficiente para ajustar modelos de lenguaje."},
    {"texto": "Pregunta: ¿Para qué usamos Colab?\nRespuesta: Para entrenar modelos usando GPUs gratuitas en la nube."},
    {"texto": "Pregunta: ¿Qué es Hugging Face?\nRespuesta: Una plataforma para encontrar y compartir modelos de IA."},
    {"texto": "Pregunta: ¿Qué hace el tokenizer?\nRespuesta: Convierte las palabras en números que el modelo puede entender."}
]

dataset = Dataset.from_list(ejemplos)

**2. Prueba el modelo base:** Genera una respuesta con el modelo sin ajustar para un prompt de prueba y guárdala en `respuesta_base`.

In [21]:
# Probar el modelo base con un prompt de prueba y guardar el resultado en respuesta_base
prompt_prueba = "Pregunta: ¿Qué es LoRA?\nRespuesta:"

respuesta_base = generar_respuesta(prompt_prueba, modelo, tokenizer)
print("Respuesta del modelo sin ajustar:\n", respuesta_base)

Respuesta del modelo sin ajustar:
 Pregunta: ¿Qué es LoRA?
Respuesta: LoRA es un sistema de transmisión de datos en la radiofrecuencia de 900 MHz y 2.4 GHz. Es un sistema de transmisión de datos en la radiofrecuencia de 9


**3. Configura y aplica LoRA:** Fija una semilla con `set_seed` y define tu `LoraConfig` (rango, alpha, módulos objetivo, dropout en 0) y aplícalo al modelo base.

In [22]:
# Configurar LoraConfig y aplicarlo al modelo base
from transformers import set_seed
from peft import LoraConfig, get_peft_model

set_seed(42)

config_lora = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.0,
    bias="none",
    task_type="CAUSAL_LM"
)

modelo_lora = get_peft_model(modelo, config_lora)

**4. Entrena:** Configura el `SFTTrainer` con tu dataset y ejecuta el fine-tuning; guarda la pérdida final en `perdida_final`.

In [23]:
# Configurar el Trainer con el dataset y ejecutar el fine-tuning; guardar la pérdida final en perdida_final
from trl import SFTTrainer, SFTConfig

config_entrenamiento = SFTConfig(
    output_dir="/content/resultados",
    num_train_epochs=30,
    per_device_train_batch_size=4,
    learning_rate=2e-4,
    logging_steps=1, # Para registrar la pérdida en cada paso
    dataset_text_field="texto",
    max_length=128,
    report_to="none",
    fp16=True,
    bf16=False
)

trainer = SFTTrainer(
    model=modelo_lora,
    train_dataset=dataset,
    args=config_entrenamiento,
)

# Iniciar entrenamiento
resultado_entrenamiento = trainer.train()
perdida_final = resultado_entrenamiento.training_loss

# Extraer la pérdida del primer paso para usarla como inicial
historial = trainer.state.log_history
perdida_inicial = historial[0]['loss']

Adding EOS to train dataset:   0%|          | 0/4 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/4 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/4 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/4 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/4 [00:00<?, ? examples/s]

{'loss': '3.04', 'grad_norm': '2.183', 'learning_rate': '0.0002', 'entropy': '2.394', 'mean_token_accuracy': '0.413', 'num_tokens': '142', 'epoch': '1'}
{'loss': '2.974', 'grad_norm': '2.083', 'learning_rate': '0.0001933', 'entropy': '2.392', 'mean_token_accuracy': '0.4203', 'num_tokens': '284', 'epoch': '2'}
{'loss': '2.897', 'grad_norm': '2.159', 'learning_rate': '0.0001867', 'entropy': '2.389', 'mean_token_accuracy': '0.413', 'num_tokens': '426', 'epoch': '3'}
{'loss': '2.813', 'grad_norm': '2.263', 'learning_rate': '0.00018', 'entropy': '2.385', 'mean_token_accuracy': '0.4275', 'num_tokens': '568', 'epoch': '4'}
{'loss': '2.722', 'grad_norm': '2.354', 'learning_rate': '0.0001733', 'entropy': '2.381', 'mean_token_accuracy': '0.4275', 'num_tokens': '710', 'epoch': '5'}
{'loss': '2.628', 'grad_norm': '2.414', 'learning_rate': '0.0001667', 'entropy': '2.378', 'mean_token_accuracy': '0.4275', 'num_tokens': '852', 'epoch': '6'}
{'loss': '2.535', 'grad_norm': '2.438', 'learning_rate': '0.

**5. Compara y concluye:** Calcula la reducción entre la pérdida inicial y `perdida_final` como métrica objetiva, y usa la respuesta generada por el modelo ya ajustado solo como referencia cualitativa.

In [24]:
# Comparar la perdida inicial y final del entrenamiento como metrica objetiva
reduccion = perdida_inicial - perdida_final
print(f"Pérdida inicial: {perdida_inicial:.4f}")
print(f"Pérdida final: {perdida_final:.4f}")
print(f"Mejora (reducción): {reduccion:.4f}")

Pérdida inicial: 3.0400
Pérdida final: 1.9933
Mejora (reducción): 1.0467


In [26]:
# Como referencia cualitativa (puede variar de sesión a sesión con un dataset tan chico):
respuesta_ajustada = generar_respuesta(prompt_prueba, modelo_lora, tokenizer)

print("ANTES DEL FINE-TUNING:\n", respuesta_base)
print("\n--------------------------\n")
print("DESPUÉS DEL FINE-TUNING:\n", respuesta_ajustada)

ANTES DEL FINE-TUNING:
 Pregunta: ¿Qué es LoRA?
Respuesta: LoRA es un sistema de transmisión de datos en la radiofrecuencia de 900 MHz y 2.4 GHz. Es un sistema de transmisión de datos en la radiofrecuencia de 9

--------------------------

DESPUÉS DEL FINE-TUNING:
 Pregunta: ¿Qué es LoRA?
Respuesta: Esper and the 













































